# LinkShield-AI — Phase 1 End-to-End

**Purpose:** Reproduce the Phase 1 phishing URL baseline from the official UCI dataset.

**Dataset:** PhiUSIIL Phishing URL (Website), UCI ID 967.

**Method:** URL-only input, Person 2 cleaning, stratified 80/20 split (seed 42), and Person 3 Logistic Regression Pipeline.

**Safety:** URLs are treated as inert text. The notebook downloads the dataset but never visits or resolves individual URLs.

In [15]:
%pip install -q ucimlrepo
!git clone -q https://github.com/Remonda230105823/LinkShield-AI.git
%cd LinkShield-AI

/content/LinkShield-AI/LinkShield-AI


## 1. Project setup

In a fresh Google Colab runtime, upload the project's `artifacts.zip` if the `src` folder is not already available. The ZIP must contain the project's `src` folder.

In [ ]:
import sys
import json
import tempfile
from pathlib import Path
from zipfile import ZipFile

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates if (p / "src").is_dir()),
    None
)

if PROJECT_ROOT is None:
    try:
        from google.colab import files

        print("Upload the project artifacts.zip containing the src folder.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

        if not zip_names:
            raise RuntimeError("No ZIP file was uploaded.")

        with ZipFile(zip_names[0]) as archive:
            archive.extractall("/content")

        PROJECT_ROOT = Path("/content")

    except ImportError as exc:
        raise RuntimeError(
            "Project src folder was not found. Open this notebook "
            "from the project root or run it in Google Colab and upload artifacts.zip."
        ) from exc

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("The uploaded ZIP does not contain a top-level src folder.")

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
print("Source folder found:", (PROJECT_ROOT / "src").is_dir())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Image
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

from src.data_cleaning import (
    convert_original_label,
    run_cleaning_pipeline,
)

from src.train_baseline import (
    load_split,
    verify_contract,
    cross_validate_on_train,
    train_baseline,
    evaluate,
    business_metric_inputs,
    top_coefficients,
    save_confusion_matrix_png,
    _hostname,
)

print("Imports completed.")
print("Project root:", PROJECT_ROOT)

## 2. Download the official UCI dataset

Only the raw URL and original target label are selected. The original UCI label convention is inverted by Person 2's conversion function so that `1 = phishing` and `0 = legitimate`.

In [ ]:
dataset = fetch_ucirepo(id=967)

raw_features = dataset.data.features
raw_targets = dataset.data.targets

raw_df = pd.DataFrame({
    "url": raw_features["URL"],
    "is_phishing": convert_original_label(raw_targets["label"]),
})

print("Raw dataset shape:", raw_df.shape)
print("Columns used:", raw_df.columns.tolist())
print("Label counts:")
display(raw_df["is_phishing"].value_counts().sort_index())

## 3. Clean the dataset

Use Person 2's existing cleaning pipeline. No precomputed UCI features are used.

In [ ]:
clean_df, quarantined_df, cleaning_report = run_cleaning_pipeline(raw_df)

print("Raw rows:", len(raw_df))
print("Clean rows:", len(clean_df))
print("Quarantined rows:", len(quarantined_df))
print("Clean columns:", clean_df.columns.tolist())

print("\nClass distribution:")
display(
    clean_df["is_phishing"]
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

print("\nCleaning report:")
print(json.dumps(cleaning_report, indent=2, default=str))

## 4. Create the agreed train/test split

The split uses stratification, 20% test data, and random seed 42. Temporary files are stored separately so existing team data and artifacts are not overwritten.

In [ ]:
RANDOM_SEED = 42
TEST_SIZE = 0.20

train_df, test_df = train_test_split(
    clean_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=clean_df["is_phishing"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

RUN_ROOT = Path(tempfile.mkdtemp(prefix="linkshield_e2e_"))
processed_dir = RUN_ROOT / "data" / "processed"
artifacts_dir = RUN_ROOT / "artifacts"
reports_dir = RUN_ROOT / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

clean_df.reset_index(drop=True).to_csv(
    processed_dir / "cleaned_urls.csv", index=False
)
train_df.to_csv(processed_dir / "train.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

split_manifest = {
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "random_seed": RANDOM_SEED,
    "train_class_counts": {
        "legitimate (0)": int((train_df["is_phishing"] == 0).sum()),
        "phishing (1)": int((train_df["is_phishing"] == 1).sum()),
    },
    "test_class_counts": {
        "legitimate (0)": int((test_df["is_phishing"] == 0).sum()),
        "phishing (1)": int((test_df["is_phishing"] == 1).sum()),
    },
}

with (artifacts_dir / "split_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Temporary run directory:", RUN_ROOT)

## 5. Load the split and verify the modelling contract

Person 3's original verification function is used. Its supporting CSV and manifest files are created only inside the temporary run directory.

In [ ]:
X_train, y_train, X_test, y_test = load_split(project_root=RUN_ROOT)

contract_checks = verify_contract(
    X_train,
    y_train,
    X_test,
    y_test,
    project_root=RUN_ROOT,
)

display(pd.Series(contract_checks, name="Result").to_frame())
print("Contract verification passed.")

## 6. Cross-validation and baseline training

Five-fold cross-validation is performed on training data only. The Logistic Regression Pipeline is then fitted on the full training split.

In [ ]:
cv_results = cross_validate_on_train(X_train, y_train)

pipeline, training_info = train_baseline(X_train, y_train)

print("Training information:")
print(json.dumps(training_info, indent=2, default=str))

print("\nTraining-only cross-validation:")
print(json.dumps(cv_results, indent=2, default=str))

## 7. Evaluate on the held-out test set

The test set is evaluated once at the agreed threshold. Hostname overlap is measured as a diagnostic, not used to create a new split.

In [ ]:
train_hostnames = set(X_train.map(_hostname))

metrics = evaluate(
    pipeline,
    X_test,
    y_test,
    train_hostnames=train_hostnames,
)

main_metrics = {
    key: metrics[key]
    for key in [
        "precision_phishing",
        "recall_phishing",
        "f1_phishing",
        "accuracy",
        "roc_auc",
        "average_precision",
        "false_positive_rate",
        "false_negative_rate",
        "tn", "fp", "fn", "tp",
        "test_rows",
    ]
}

display(pd.Series(main_metrics, name="Value").to_frame())

print("\nUnseen-hostname subset:")
print(json.dumps(metrics.get("unseen_hostname_subset", {}), indent=2))

## 8. Confusion matrix

Rows represent actual classes and columns represent predicted classes. Class 0 is legitimate and class 1 is phishing.

In [ ]:
confusion_path = reports_dir / "phase1_confusion_matrix.png"
save_confusion_matrix_png(metrics, confusion_path)

display(Image(filename=str(confusion_path)))

## 9. Business metric — estimated analyst review time

The calculation uses the test-set class mix scaled to 10,000 screened URLs. Review times of 1, 3, and 5 minutes per alert are illustrative assumptions, not measured bank figures.

False negatives are reported separately; they are not counted as alerts.

In [ ]:
business_metrics = business_metric_inputs(metrics)

print("Basis:", business_metrics["basis"])
print("Alerts (TP + FP):", round(
    business_metrics["alerts_raised_tp_plus_fp"], 1
))
print("False alarms (FP):", round(
    business_metrics["of_which_false_alarms_fp"], 1
))
print("Missed phishing (FN):", round(
    business_metrics["missed_phishing_fn"], 1
))

review_table = pd.DataFrame([
    {
        "Assumed review time": label,
        "Estimated analyst hours": hours,
    }
    for label, hours in
    business_metrics["review_hours_by_assumed_minutes_per_alert"].items()
])

display(review_table)
print(business_metrics["assumption_note"])

## 10. Model coefficient sanity check

These coefficients provide a basic view of the Logistic Regression model. They are not SHAP or LIME explanations.

In [ ]:
coefficients = top_coefficients(pipeline)

print("Features pushing towards phishing:")
display(pd.DataFrame(coefficients["numeric_pushing_towards_phishing"]))

print("Features pushing towards legitimate:")
display(pd.DataFrame(coefficients["numeric_pushing_towards_legitimate"]))

## 11. Save the run results

Outputs are saved in the temporary run directory. Existing project artifacts are not replaced.

In [ ]:
results = {
    "contract_checks": contract_checks,
    "training_info": training_info,
    "cross_validation": cv_results,
    "test_metrics": metrics,
    "business_metric": business_metrics,
    "top_coefficients": coefficients,
}

results_path = reports_dir / "e2e_results.json"
with results_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

print("Results saved to:", results_path)
print("Confusion matrix saved to:", confusion_path)
print("Temporary run directory:", RUN_ROOT)

## 12. Limitations and safety

- The dataset is not representative of actual bank traffic.
- Analyst review times are illustrative assumptions.
- The model is a Phase 1 baseline, not a production banking system.
- False negatives represent missed phishing URLs and must be considered separately.
- URLs are processed as text only. The notebook does not visit, resolve, or interact with URL destinations.
- The hostname overlap diagnostic indicates a limitation in the evaluation split.